# Assignment 3 — RAG Walkthrough

A guided commentary version of `assignment_03_rag/`. We'll build the same mini-RAG pipeline cell by cell so you can see what each piece does in isolation.

**Prerequisites:**
- `pip install -r requirements.txt`
- `OPENAI_API_KEY` set in `.env`

Run cells top-to-bottom. The notebook reuses the *solution* modules — so it works even if you haven't filled in the starter files yet.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != 'openai-rag-evals-bootcamp' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
print('repo root:', REPO_ROOT)

## Step 1 — Look at the data

Before writing any retrieval code, eyeball the corpus. *Always* do this. You can't evaluate retrieval quality on a corpus you've never read.

In [ ]:
docs_dir = REPO_ROOT / 'data' / 'docs'
for p in sorted(docs_dir.glob('*.md')):
    print(f'{p.name:40} {p.stat().st_size:6d} bytes')

In [ ]:
print((docs_dir / '02-leave-policy.md').read_text()[:600])

## Step 2 — Chunking

We want to split each doc into pieces small enough that an embedding 'is about one thing', but large enough to contain self-sufficient context.

Compare two splits to develop intuition.

In [ ]:
import importlib
ingest = importlib.import_module('assignment_03_rag.solution.ingest')

text = (docs_dir / '02-leave-policy.md').read_text()
small = ingest.chunk_text(text, chunk_size=250, chunk_overlap=25)
large = ingest.chunk_text(text, chunk_size=1000, chunk_overlap=100)

print(f'small (250): {len(small)} chunks')
print(f'large (1000): {len(large)} chunks')
print()
print('--- first SMALL chunk ---')
print(small[0])
print()
print('--- first LARGE chunk ---')
print(large[0])

Notice how 'small' fragments related rules across chunks, while 'large' bundles too many distinct rules into one chunk. The 'right' size is empirical — Assignment 4 will measure it.

## Step 3 — Build the index

Run the full ingest. This embeds every chunk via the OpenAI API and persists to a local Chroma DB.

Cost: < $0.01.

In [ ]:
n = ingest.build_index(
    docs_dir=docs_dir,
    chroma_dir=REPO_ROOT / 'chroma_db',
    chunk_size=500,
    chunk_overlap=50,
)
print(f'Indexed {n} chunks.')

## Step 4 — Retrieve

Now we can ask the index for the top-k chunks for a question. *Always* sanity-check retrieval before wiring up generation — it's the most common source of failures.

In [ ]:
retrieve_mod = importlib.import_module('assignment_03_rag.solution.retrieve')

for q in ['How much PTO do I get?', 'What VPN client should I use?', 'What is the CEO\'s salary?']:
    print(f'\n=== {q} ===')
    for h in retrieve_mod.retrieve(q, k=3):
        print(f'  {h.source_path:35s} chunk {h.chunk_index} score={h.similarity:.3f}')

Notice the **third question is out-of-scope** — the top similarity is much lower (~0.2). That gap is what the threshold guardrail exploits.

## Step 5 — Generate a grounded answer

Now compose retrieval + generation. Pay attention to the system prompt — that's where the 'rules' of grounding and refusal live.

In [ ]:
generate_mod = importlib.import_module('assignment_03_rag.solution.generate')
print(generate_mod.SYSTEM_PROMPT)

In [ ]:
def ask(question: str, k: int = 5):
    chunks = retrieve_mod.retrieve(question, k=k)
    ans = generate_mod.answer(question, chunks)
    print(f'Q: {question}')
    print(f'\nA: {ans.text}')
    print(f'\ngrounded: {ans.is_grounded}')
    print(f'citations: {ans.citations}')

ask('How much PTO do I get?')

In [ ]:
ask('What is Acme Corp\'s stock ticker symbol?')

In [ ]:
ask('If I lose my work laptop while traveling abroad, who do I report it to and within what timeframe?')

## What to do next

1. Run **at least 10 questions of your own**. Note which ones fail and why.
2. Try toggling the threshold (e.g., `generate_mod.SIMILARITY_THRESHOLD = 0.5`) and re-running. What gets refused that shouldn't?
3. Move on to `notebooks/04-evals-walkthrough.ipynb` — that's where we *measure* all of this rigorously.